# ESIBD Pump Control

This notebook provides control and monitoring for the 5 ESIBD vacuum pumps.

## Equipment Configuration:
- **HiScroll Pumps**: 2 units (HiScroll1, HiScroll2) as backing pumps
- **HiPace300 Pumps**: 2 units (Transfer Chamber, Deposition Chamber)
- **HiPace450 Pump**: 1 unit (main high vacuum pump)

## Features:
- Individual pump control (turn on/off each pump separately)
- Live monitoring with multiple synchronized plots
- Temperature monitoring (HiScrolls and HiPaces separately)
- Data logging with timestamps
- Shared logging system using loguru

## 1. Import Required Libraries and Setup

In [ ]:
import sys
import threading
import time
from datetime import datetime
import asyncio
from typing import Dict, Optional, Any
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
import pickle
from collections import deque

from loguru import logger
import os
from pathlib import Path

# Add path to src modules
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'src'))

# Import device modules
from devices.pfeiffer.hiscroll12.hiscroll12 import HiScroll12
from devices.pfeiffer.hipacebus import HiPace300Bus

print("All libraries and device modules imported successfully")

## 2. Setup Shared Logging System

In [ ]:
# Get the repository root and create shared logs directory
repo_root = Path(os.getcwd()).parent.parent
log_dir = repo_root / "debugging" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

# Create shared log file for all devices
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
shared_log_file = log_dir / f"esibd_pumps_{timestamp}.log"

# Configure shared logger
logger.remove()  # Remove default logger

# Add console logger with INFO level
logger.add(sys.stderr, level="INFO", format="{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}")

# Add shared file logger with DEBUG level
logger.add(
    str(shared_log_file),
    level="DEBUG",
    format="{time:YYYY-MM-DD HH:mm:ss.SSS} | {level} | {name}:{function}:{line} | {message}",
    rotation="1 day",
    retention="30 days",
    compression="zip"
)

logger.info("ESIBD Pumps system initialized with shared logging")
print(f"Repository root: {repo_root}")
print(f"Shared logs will be saved to: {shared_log_file}")

## 3. Device Configuration (COM Ports & Addresses)

In [ ]:
# Device configuration dictionaries
# Modify these COM ports and addresses according to your actual hardware setup

#HiScroll backing pumps
HiScroll1_dict = {
    'com_port': 'COM26',
    'device_address': 2,
    'device_type': 'HiScroll12',
    'description': 'Backing pump 1'
}

HiScroll2_dict = {
    'com_port': 'COM27',
    'device_address': 2,
    'device_type': 'HiScroll12',
    'description': 'Backing pump 2'
}

# HiPace300 turbo pumps
HiPace300_Transfer_dict = {
    'com_port': 'COM21',
    'device_address': 101,
    'omnicontrol_address': 101,
    'tc400_address': 1,
    'gauge1_address': 122,
    'device_type': 'HiPace300',
    'description': 'Transfer chamber turbo pump'
}

HiPace300_Depo_dict = {
    'com_port': 'COM24',
    'device_address': 101,
    'omnicontrol_address': 101,
    'tc400_address': 1,
    'gauge1_address': 122,
    'device_type': 'HiPace300', 
    'description': 'Deposition chamber turbo pump'
}

# HiPace450 main turbo pump (placeholder - needs device class implementation)
HiPace450_dict = {
    'com_port': 'COM22',
    'device_address': 101,
    'omnicontrol_address': 101,
    'tc400_address': 1,
    'gauge1_address': 122,
    'device_type': 'HiPace450',
    'description': 'Main high vacuum pump (initialized as HiPace300Bus)'
}

# Print configuration summary
print("=" * 60)
print("ESIBD PUMPS - DEVICE CONFIGURATION")
print("=" * 60)
print(f"\nHiScroll1: {HiScroll1_dict['com_port']} @ Address {HiScroll1_dict['device_address']}")
print(f"HiScroll2: {HiScroll2_dict['com_port']} @ Address {HiScroll2_dict['device_address']}")
print(f"HiPace300 Transfer: {HiPace300_Transfer_dict['com_port']} @ Address {HiPace300_Transfer_dict['device_address']}")
print(f"HiPace300 Depo: {HiPace300_Depo_dict['com_port']} @ Address {HiPace300_Depo_dict['device_address']}")
print(f"HiPace450: {HiPace450_dict['com_port']} @ Address {HiPace450_dict['device_address']} (placeholder)")
print("\nDevice dictionaries configured successfully")

## 4. Connect Devices

In [ ]:
# Initialize and connect HiScroll1
hiscroll1 = HiScroll12(
    device_id="hiscroll1",
    port=HiScroll1_dict['com_port'],
    device_address=HiScroll1_dict['device_address'],
    logger=logger
)
hiscroll1.connect()
print(f"HiScroll1 connected on {HiScroll1_dict['com_port']}")

In [ ]:
# Initialize and connect HiScroll2
hiscroll2 = HiScroll12(
    device_id="hiscroll2",
    port=HiScroll2_dict['com_port'],
    device_address=HiScroll2_dict['device_address'],
    logger=logger
)
hiscroll2.connect()
print(f"HiScroll2 connected on {HiScroll2_dict['com_port']}")

In [ ]:
# Initialize and connect HiPace300 Transfer
hipace_transfer = HiPace300Bus(
    device_id="hipace_transfer",
    port=HiPace300_Transfer_dict['com_port'],
    device_address=HiPace300_Transfer_dict['device_address'],
    omnicontrol_address=HiPace300_Transfer_dict['omnicontrol_address'],
    tc400_address=HiPace300_Transfer_dict['tc400_address'],
    gauge1_address=HiPace300_Transfer_dict['gauge1_address'],
    logger=logger
)
hipace_transfer.connect()
print(f"HiPace300 Transfer connected on {HiPace300_Transfer_dict['com_port']}")

In [ ]:
# Initialize and connect HiPace300 Depo
hipace_depo = HiPace300Bus(
    device_id="hipace_depo",
    port=HiPace300_Depo_dict['com_port'],
    device_address=HiPace300_Depo_dict['device_address'],
    omnicontrol_address=HiPace300_Depo_dict['omnicontrol_address'],
    tc400_address=HiPace300_Depo_dict['tc400_address'],
    gauge1_address=HiPace300_Depo_dict['gauge1_address'],
    logger=logger
)
hipace_depo.connect()
print(f"HiPace300 Depo connected on {HiPace300_Depo_dict['com_port']}")

In [ ]:
# Initialize and connect HiPace450 (using HiPace300Bus class)
hipace450 = HiPace300Bus(
    device_id="hipace450",
    port=HiPace450_dict['com_port'],
    device_address=HiPace450_dict['device_address'],
    omnicontrol_address=HiPace450_dict['omnicontrol_address'],
    tc400_address=HiPace450_dict['tc400_address'],
    gauge1_address=HiPace450_dict['gauge1_address'],
    logger=logger
)
hipace450.connect()
print(f"HiPace450 connected on {HiPace450_dict['com_port']} (initialized as HiPace300Bus)")

In [ ]:
print(hipace450.get_operating_hours_pump())
print(hipace_depo.get_operating_hours_pump())
print(hipace_transfer.get_operating_hours_pump())

In [ ]:
hiscroll1.get_electronics_name()

## 5. Start Housekeeping for All Devices

In [ ]:
# Start housekeeping for all pumps
hiscroll1.start_housekeeping()
hiscroll2.start_housekeeping()
hipace_transfer.start_housekeeping()
hipace_depo.start_housekeeping()
hipace450.start_housekeeping()

print("Housekeeping started for all devices:")
print("  - HiScroll1")
print("  - HiScroll2")
print("  - HiPace300 Transfer")
print("  - HiPace300 Depo")
print("  - HiPace450")

## 5.1 Stop Housekeeping

In [ ]:
hiscroll1.stop_housekeeping()
hiscroll2.stop_housekeeping()
hipace_transfer.stop_housekeeping()
hipace_depo.stop_housekeeping()
hipace450.stop_housekeeping()

## 6. Individual Pump Control

This section provides control cells for turning on/off each pump individually.

### 6.1 HiScroll Pump Control

In [ ]:
# Enable HiScroll1
hiscroll1.enable_pump()
print("HiScroll1 enabled")

In [ ]:
# Disable HiScroll1
hiscroll1.disable_pump()
print("HiScroll1 disabled")

In [ ]:
# Enable HiScroll2
hiscroll2.enable_pump()
print("HiScroll2 enabled")

In [ ]:
# Disable HiScroll2
hiscroll2.disable_pump()
print("HiScroll2 disabled")

### 6.2 HiPace300 Pump Control ALWAYS BOTH AT THE SAME TIME

In [ ]:
# Enable HiPace300 Transfer pump
hipace_transfer.enable_motor_pump()
# Enable HiPace300 Depo pump
hipace_depo.enable_motor_pump()

In [ ]:
hipace_transfer.enable_pumpStatn()
hipace_depo.enable_pumpStatn()
print("HiPace300 pumps enabled")

In [ ]:
# Disable HiPace300 Depo pump
hipace_depo.disable_pumpStatn()
print("HiPace300 Depo pump disabled")

# Disable HiPace300 Transfer pump
hipace_transfer.disable_pumpStatn()
print("HiPace300 Transfer pump disabled")

### 6.3 HiPace450 Pump Control

In [ ]:
hipace450.get_operating_hours_pump()

In [ ]:
# Enable HiPace450 pump
hipace450.enable_motor_pump()

print("HiPace450 pump enabled")

In [ ]:
hipace450.enable_pumpStatn()

In [ ]:
# Disable HiPace450 pump
hipace450.disable_pumpStatn()
print("HiPace450 pump disabled")

## 7. Live Plotting - Pump Monitoring

This section provides live plotting for pump temperature, RPM, and power monitoring.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np
from collections import deque

class PumpChainPlotter:
    """Live plotting class for ESIBD pump monitoring"""
    
    def __init__(self, update_interval=1000):
        self.update_interval = update_interval
        
        # Data storage (keep last 100 points)
        self.timestamps = deque(maxlen=100)
        
        # Temperature data for HiScrolls
        self.hiscroll1_temp = deque(maxlen=100)
        self.hiscroll2_temp = deque(maxlen=100)
        
        # Temperature data for HiPaces
        self.hipace_transfer_temp = deque(maxlen=100)
        self.hipace_depo_temp = deque(maxlen=100)
        self.hipace450_temp = deque(maxlen=100)
        
        # RPM data for HiPaces
        self.hipace_transfer_rpm = deque(maxlen=100)
        self.hipace_depo_rpm = deque(maxlen=100)
        self.hipace450_rpm = deque(maxlen=100)
        
        # Power data for HiPaces
        self.hipace_transfer_power = deque(maxlen=100)
        self.hipace_depo_power = deque(maxlen=100)
        self.hipace450_power = deque(maxlen=100)
        
        # Create figure with 4 rows
        self.fig, (self.ax_temp_hiscroll, self.ax_temp_hipace,
                   self.ax_hipace_rpm, self.ax_hipace_power) = plt.subplots(
            4, 1, figsize=(12, 12), sharex=True
        )
        self.fig.suptitle('ESIBD Pump Monitoring', fontsize=14)
    
    def update_plot(self, frame):
        try:
            current_time = datetime.now()
            self.timestamps.append(current_time)
            
            # Collect HiScroll temperature data
            try:
                self.hiscroll1_temp.append(hiscroll1.get_temp_motor())
            except:
                self.hiscroll1_temp.append(None)
            
            try:
                self.hiscroll2_temp.append(hiscroll2.get_temp_motor())
            except:
                self.hiscroll2_temp.append(None)
            
            # Collect HiPace temperature data
            try:
                self.hipace_transfer_temp.append(hipace_transfer.get_motor_temperature())
            except:
                self.hipace_transfer_temp.append(None)
            
            try:
                self.hipace_depo_temp.append(hipace_depo.get_motor_temperature())
            except:
                self.hipace_depo_temp.append(None)
            
            try:
                self.hipace450_temp.append(hipace450.get_motor_temperature())
            except:
                self.hipace450_temp.append(None)
            
            # Collect HiPace RPM data
            try:
                self.hipace_transfer_rpm.append(hipace_transfer.get_actual_speed_rpm())
            except:
                self.hipace_transfer_rpm.append(None)
            
            try:
                self.hipace_depo_rpm.append(hipace_depo.get_actual_speed_rpm())
            except:
                self.hipace_depo_rpm.append(None)
            
            try:
                self.hipace450_rpm.append(hipace450.get_actual_speed_rpm())
            except:
                self.hipace450_rpm.append(None)
            
            # Collect HiPace Power data
            try:
                self.hipace_transfer_power.append(hipace_transfer.get_drive_power())
            except:
                self.hipace_transfer_power.append(None)
            
            try:
                self.hipace_depo_power.append(hipace_depo.get_drive_power())
            except:
                self.hipace_depo_power.append(None)
            
            try:
                self.hipace450_power.append(hipace450.get_drive_power())
            except:
                self.hipace450_power.append(None)
            
        except Exception as e:
            logger.warning(f"Data collection error: {e}")
            return
        
        # Clear all axes
        self.ax_temp_hiscroll.clear()
        self.ax_temp_hipace.clear()
        self.ax_hipace_rpm.clear()
        self.ax_hipace_power.clear()
        
        # Convert deques to lists
        times = list(self.timestamps)
        
        if not times:
            return
        
        try:
            # Plot HiScroll Temperatures
            hs1_temps = list(self.hiscroll1_temp)
            hs2_temps = list(self.hiscroll2_temp)
            
            valid_hs1 = [(t, temp) for t, temp in zip(times, hs1_temps) if temp is not None]
            valid_hs2 = [(t, temp) for t, temp in zip(times, hs2_temps) if temp is not None]
            
            if valid_hs1:
                t_hs1, temp_hs1 = zip(*valid_hs1)
                self.ax_temp_hiscroll.plot(t_hs1, temp_hs1, 'b-', marker='o', label='HiScroll1')
            
            if valid_hs2:
                t_hs2, temp_hs2 = zip(*valid_hs2)
                self.ax_temp_hiscroll.plot(t_hs2, temp_hs2, 'r-', marker='s', label='HiScroll2')
            
            self.ax_temp_hiscroll.set_ylabel('Temperature [degC]')
            self.ax_temp_hiscroll.set_title('HiScroll Pump Temperatures')
            self.ax_temp_hiscroll.legend(loc='upper right')
            self.ax_temp_hiscroll.grid(True)
            
            # Plot HiPace Temperatures
            hp_t_temps = list(self.hipace_transfer_temp)
            hp_d_temps = list(self.hipace_depo_temp)
            hp_450_temps = list(self.hipace450_temp)
            
            valid_hp_t = [(t, temp) for t, temp in zip(times, hp_t_temps) if temp is not None]
            valid_hp_d = [(t, temp) for t, temp in zip(times, hp_d_temps) if temp is not None]
            valid_hp_450 = [(t, temp) for t, temp in zip(times, hp_450_temps) if temp is not None]
            
            if valid_hp_t:
                t_hp_t, temp_hp_t = zip(*valid_hp_t)
                self.ax_temp_hipace.plot(t_hp_t, temp_hp_t, 'g-', marker='^', label='HiPace Transfer')
            
            if valid_hp_d:
                t_hp_d, temp_hp_d = zip(*valid_hp_d)
                self.ax_temp_hipace.plot(t_hp_d, temp_hp_d, 'm-', marker='d', label='HiPace Depo')
            
            if valid_hp_450:
                t_hp_450, temp_hp_450 = zip(*valid_hp_450)
                self.ax_temp_hipace.plot(t_hp_450, temp_hp_450, 'orange', marker='s', label='HiPace450')
            
            self.ax_temp_hipace.set_ylabel('Temperature [degC]')
            self.ax_temp_hipace.set_title('HiPace Pump Temperatures')
            self.ax_temp_hipace.legend(loc='upper right')
            self.ax_temp_hipace.grid(True)
            
            # Plot HiPace RPM
            rpm_t = list(self.hipace_transfer_rpm)
            rpm_d = list(self.hipace_depo_rpm)
            rpm_450 = list(self.hipace450_rpm)
            
            valid_rpm_t = [(t, rpm) for t, rpm in zip(times, rpm_t) if rpm is not None]
            valid_rpm_d = [(t, rpm) for t, rpm in zip(times, rpm_d) if rpm is not None]
            valid_rpm_450 = [(t, rpm) for t, rpm in zip(times, rpm_450) if rpm is not None]
            
            if valid_rpm_t:
                t_rpm_t, rpm_t_val = zip(*valid_rpm_t)
                self.ax_hipace_rpm.plot(t_rpm_t, rpm_t_val, 'g-', marker='^', label='HiPace Transfer')
            
            if valid_rpm_d:
                t_rpm_d, rpm_d_val = zip(*valid_rpm_d)
                self.ax_hipace_rpm.plot(t_rpm_d, rpm_d_val, 'm-', marker='d', label='HiPace Depo')
            
            if valid_rpm_450:
                t_rpm_450, rpm_450_val = zip(*valid_rpm_450)
                self.ax_hipace_rpm.plot(t_rpm_450, rpm_450_val, 'orange', marker='s', label='HiPace450')
            
            self.ax_hipace_rpm.set_ylabel('RPM')
            self.ax_hipace_rpm.set_title('HiPace Pump Rotation Speed')
            self.ax_hipace_rpm.legend(loc='upper right')
            self.ax_hipace_rpm.grid(True)
            
            # Plot HiPace Power
            power_t = list(self.hipace_transfer_power)
            power_d = list(self.hipace_depo_power)
            power_450 = list(self.hipace450_power)
            
            valid_power_t = [(t, p) for t, p in zip(times, power_t) if p is not None]
            valid_power_d = [(t, p) for t, p in zip(times, power_d) if p is not None]
            valid_power_450 = [(t, p) for t, p in zip(times, power_450) if p is not None]
            
            if valid_power_t:
                t_power_t, power_t_val = zip(*valid_power_t)
                self.ax_hipace_power.plot(t_power_t, power_t_val, 'g-', marker='^', label='HiPace Transfer')
            
            if valid_power_d:
                t_power_d, power_d_val = zip(*valid_power_d)
                self.ax_hipace_power.plot(t_power_d, power_d_val, 'm-', marker='d', label='HiPace Depo')
            
            if valid_power_450:
                t_power_450, power_450_val = zip(*valid_power_450)
                self.ax_hipace_power.plot(t_power_450, power_450_val, 'orange', marker='s', label='HiPace450')
            
            self.ax_hipace_power.set_ylabel('Power [W]')
            self.ax_hipace_power.set_xlabel('Time')
            self.ax_hipace_power.set_title('HiPace Pump Drive Power')
            self.ax_hipace_power.legend(loc='upper right')
            self.ax_hipace_power.grid(True)
            
        except Exception as e:
            logger.error(f"Plotting error: {e}")
        
        # Format x-axis for all plots
        for ax in [self.ax_temp_hiscroll, self.ax_temp_hipace,
                   self.ax_hipace_rpm, self.ax_hipace_power]:
            ax.tick_params(axis='x', rotation=45)
        
        # Hide x-tick labels for all except bottom
        for ax in [self.ax_temp_hiscroll, self.ax_temp_hipace, self.ax_hipace_rpm]:
            plt.setp(ax.get_xticklabels(), visible=False)
        
        self.fig.tight_layout()
    
    def start_live_plot(self):
        """Start live plotting"""
        self.ani = FuncAnimation(
            self.fig,
            self.update_plot,
            interval=self.update_interval,
            blit=False,
            cache_frame_data=False
        )
        
        plt.show()
        print("Live plotting started - Close window to stop")
    
    def stop_live_plot(self):
        """Stop live plotting"""
        if hasattr(self, 'ani'):
            self.ani.event_source.stop()
        plt.close(self.fig)
        print("Live plotting stopped.")

    def save_plot(self, filename='pump_chain_plot.png', dpi=150):
        """Save current plot to file"""
        self.fig.savefig(filename, dpi=dpi, bbox_inches='tight')
        print(f"Plot saved to {filename}")

# Create plotter instance
plotter = PumpChainPlotter(update_interval=1000)
print("Pump chain plotter ready")
print("   - HiScroll pump temperatures")
print("   - HiPace pump temperatures")
print("   - HiPace pump RPM")
print("   - HiPace pump power")

In [ ]:
print(hipace_depo.get_pump_error_code())
print(hipace_transfer.get_pump_error_code())
print(hipace450.get_pump_error_code())

In [ ]:
%matplotlib notebook

In [ ]:
# Create and start live plotting
plotter = PumpChainPlotter(update_interval=2000)  # Update every 2 seconds
plotter.start_live_plot()

In [ ]:
plotter.save_plot('Plot2.png', dpi=600)

In [ ]:
# Stop live plotting
plotter.stop_live_plot()

## 8. Data Logging and Export

Export collected data for further analysis.

In [ ]:
def export_pump_data(plotter, filename=None):
    """Export collected data to CSV file"""
    
    if filename is None:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"esibd_pumps_{timestamp}.csv"
    
    # Create DataFrame from collected data
    data = {
        'timestamp': list(plotter.timestamps),
        'hiscroll1_temp': list(plotter.hiscroll1_temp),
        'hiscroll2_temp': list(plotter.hiscroll2_temp),
        'hipace_transfer_temp': list(plotter.hipace_transfer_temp),
        'hipace_depo_temp': list(plotter.hipace_depo_temp),
        'hipace450_temp': list(plotter.hipace450_temp),
        'hipace_transfer_rpm': list(plotter.hipace_transfer_rpm),
        'hipace_depo_rpm': list(plotter.hipace_depo_rpm),
        'hipace450_rpm': list(plotter.hipace450_rpm),
        'hipace_transfer_power': list(plotter.hipace_transfer_power),
        'hipace_depo_power': list(plotter.hipace_depo_power),
        'hipace450_power': list(plotter.hipace450_power)
    }
    
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False)
    logger.info(f"Data exported to {filename}")
    print(f"Data exported to {filename}")
    print(f"   Total data points: {len(df)}")
    return df

print("Data export function ready")

In [ ]:
# Export collected data
df = export_pump_data(plotter)

## 9. Stop Housekeeping and Disconnect Devices

In [ ]:
# Stop housekeeping for all devices
hiscroll1.stop_housekeeping()
hiscroll2.stop_housekeeping()
hipace_transfer.stop_housekeeping()
hipace_depo.stop_housekeeping()
hipace450.stop_housekeeping()

print("Housekeeping stopped for all devices")

In [ ]:
# Disconnect all devices
print("Disconnecting all devices...")

try:
    hiscroll1.disconnect()
    print("  HiScroll1 disconnected")
except Exception as e:
    print(f"  HiScroll1 disconnect error: {e}")

try:
    hiscroll2.disconnect()
    print("  HiScroll2 disconnected")
except Exception as e:
    print(f"  HiScroll2 disconnect error: {e}")

try:
    hipace_transfer.disconnect()
    print("  HiPace300 Transfer disconnected")
except Exception as e:
    print(f"  HiPace300 Transfer disconnect error: {e}")

try:
    hipace_depo.disconnect()
    print("  HiPace300 Depo disconnected")
except Exception as e:
    print(f"  HiPace300 Depo disconnect error: {e}")

try:
    hipace450.disconnect()
    print("  HiPace450 disconnected")
except Exception as e:
    print(f"  HiPace450 disconnect error: {e}")

print("\nAll devices disconnected")

## Summary & Quick Reference

### Device Overview:
- **HiScroll1 & HiScroll2**: Backing pumps
- **HiPace300 Transfer**: Transfer chamber turbo pump
- **HiPace300 Depo**: Deposition chamber turbo pump
- **HiPace450**: Main high vacuum pump

### Live Plotting Features:
- **HiScroll Temperature Plot**: Combined plot for both HiScroll pumps
- **HiPace Temperature Plot**: Combined plot for all HiPace pumps
- **HiPace RPM Plot**: Rotation speed for all HiPace pumps
- **HiPace Power Plot**: Drive power for all HiPace pumps

### Key Operations:

#### Connect All Devices:
Run cells in Section 4 sequentially

#### Start Monitoring:
1. Start housekeeping (Section 5)
2. Enable live plotting (Section 7)

#### Control Individual Pumps:
- Section 6.1: HiScroll pump control
- Section 6.2: HiPace300 pump control
- Section 6.3: HiPace450 pump control

#### Export Data:
Run export cell in Section 8

#### Shutdown:
1. Stop live plotting
2. Stop housekeeping (Section 9)
3. Disconnect all devices (Section 9)

### Notes:
- Always start backing pumps (HiScrolls) before turbo pumps (HiPaces)
- Monitor pressure before enabling turbo pumps
- Wait for pressure to be low enough before starting turbo pumps